# Module 03 — Tool Use

**Goal:** Let the LLM call your functions instead of just generating text.

In Module 02 the developer decides every step. Here the **LLM decides**:
- Which function to call
- When to call it
- What arguments to pass

```
question → LLM sees tools → calls get_schema() → calls run_query(sql=...) → writes answer
```

This notebook runs entirely on the **Databricks free tier** against the built-in `samples` catalog — no data setup needed.

---
**Run each cell in order (Shift+Enter).**

## Step 1 — Install `openai`

In [ ]:
%pip install --quiet openai
dbutils.library.restartPython()

## Step 2 — Set your parameters

Pick any schema from the built-in `samples` catalog: `bakehouse`, `nyctaxi`, `accuweather`, `tpch`, etc.

In [ ]:
dbutils.widgets.text("API_KEY", "", "API key")
dbutils.widgets.text("BASE_URL", "https://models.github.ai/inference", "Base URL")
dbutils.widgets.text("MODEL", "openai/gpt-4o-mini", "Model")
dbutils.widgets.text("CATALOG", "samples", "Catalog")
dbutils.widgets.text("SCHEMA", "bakehouse", "Schema")

API_KEY = dbutils.widgets.get("API_KEY")
BASE_URL = dbutils.widgets.get("BASE_URL")
MODEL = dbutils.widgets.get("MODEL")
CATALOG = dbutils.widgets.get("CATALOG")
SCHEMA = dbutils.widgets.get("SCHEMA")

if not API_KEY:
    raise ValueError("Paste your API key into the API_KEY widget at the top of the notebook.")

from openai import OpenAI
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Model:    {MODEL}")
print(f"Database: {CATALOG}.{SCHEMA}")

## Step 3 — Database and safety helpers

Three small helpers the agent will call:
- `get_schema()` — `SHOW TABLES` + `DESCRIBE TABLE` for every table
- `run_query(sql)` — `spark.sql(sql)` returning `(columns, rows)`
- `validate_sql(sql)` — blocks anything that isn't a single SELECT before it touches the DB

In [ ]:
import re, json
from dataclasses import dataclass

@dataclass
class ValidationResult:
    is_safe: bool
    reason: str

_FORBIDDEN = {"DROP","DELETE","UPDATE","INSERT","ALTER","TRUNCATE","CREATE","REPLACE","MERGE","EXEC","EXECUTE","GRANT","REVOKE","ATTACH","DETACH"}

def validate_sql(sql: str) -> ValidationResult:
    stripped = sql.strip()
    if not stripped:
        return ValidationResult(False, "SQL is empty.")
    cleaned = stripped.rstrip(";")
    if ";" in cleaned:
        return ValidationResult(False, "Only a single SELECT statement is allowed.")
    if cleaned.split()[0].upper() != "SELECT":
        return ValidationResult(False, f"Query must start with SELECT, got '{cleaned.split()[0]}'.")
    for kw in _FORBIDDEN:
        if re.search(rf"\b{kw}\b", cleaned.upper()):
            return ValidationResult(False, f"Forbidden keyword: {kw}")
    return ValidationResult(True, "ok")


def get_schema() -> dict:
    schema = {}
    for row in spark.sql("SHOW TABLES").collect():
        table_name = row.tableName
        cols = spark.sql(f"DESCRIBE TABLE {table_name}").collect()
        schema[table_name] = [
            {"name": c.col_name, "type": c.data_type}
            for c in cols
            if c.col_name and not c.col_name.startswith("#")
        ]
    return schema


def run_query(sql: str):
    df = spark.sql(sql).limit(50)
    columns = df.columns
    rows = [row.asDict() for row in df.collect()]
    return columns, rows


_schema = get_schema()
print(f"Tables in {CATALOG}.{SCHEMA}: {list(_schema.keys())}")

## Step 4 — Describe the tools to the LLM

`TOOLS` is a JSON schema. The LLM reads the `description` fields to decide **which** tool to call and **what** arguments to pass. Good descriptions matter a lot — they are the contract between you and the model.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_schema",
            "description": (
                "Inspect the database \u2014 returns every table name and its column "
                "definitions (name + type). Always call this first if you don't "
                "already know the schema."
            ),
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_query",
            "description": (
                "Execute a read-only SELECT query against the database and return "
                "the results as JSON. Only SELECT statements are allowed \u2014 any "
                "write operation will be blocked."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {"type": "string", "description": "A valid Spark SQL SELECT statement."}
                },
                "required": ["sql"],
            },
        },
    },
]

SYSTEM_PROMPT = f"""\
You are a helpful database assistant connected to a Spark SQL database.
The current catalog and schema are already set to `{CATALOG}.{SCHEMA}`, so use unqualified table names.

You have two tools available:
  - get_schema: inspect the database to see its tables and columns
  - run_query:  execute a SELECT query and get back rows

Strategy:
1. Always call get_schema first to understand the data before writing SQL.
2. Write and run the minimal query that answers the question.
3. After getting results, summarise them clearly for the user.

Important: only generate SELECT queries. Never attempt write operations.
"""

## Step 5 — Execute tool calls

When the LLM says "call `run_query` with sql=...", we need to actually run it and feed the result back.

In [ ]:
def run_tool(name: str, arguments: dict) -> str:
    if name == "get_schema":
        return json.dumps(get_schema(), indent=2)

    if name == "run_query":
        sql = arguments["sql"]
        validation = validate_sql(sql)
        if not validation.is_safe:
            return f"ERROR (safety check failed): {validation.reason}"
        try:
            columns, rows = run_query(sql)
            return json.dumps({"columns": columns, "rows": rows}, default=str)
        except Exception as exc:
            return f"ERROR (query failed): {exc}"

    return f"ERROR: unknown tool '{name}'"

## Step 6 — The tool-calling loop

This is the core of every tool-use agent:
1. Send messages + tools to the LLM.
2. If `finish_reason == "tool_calls"`: run the tools, append the results, go back to 1.
3. If `finish_reason == "stop"`: the LLM has its answer — print and done.

In [ ]:
def ask(question: str) -> None:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]

    print(f"\nQuestion: {question}")
    print("-" * 60)

    step = 0
    while True:
        step += 1
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            temperature=0,
        )

        choice = response.choices[0]
        message = choice.message
        messages.append(message)

        if choice.finish_reason == "stop":
            print(f"\nAnswer:\n{message.content}")
            break

        if choice.finish_reason == "tool_calls":
            for tool_call in message.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)

                print(f"[step {step}] \u2192 {fn_name}({json.dumps(fn_args) if fn_args else ''})")
                result = run_tool(fn_name, fn_args)

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })

                preview = result[:120].replace("\n", " ")
                print(f"         \u2190 {preview}{'...' if len(result) > 120 else ''}")
        else:
            print(f"Unexpected finish_reason: {choice.finish_reason}")
            break

    print("-" * 60)
    print(f"Total steps: {step}  |  Total messages: {len(messages)}")

## Step 7 — Try it

Watch the agent call `get_schema` first, then write SQL, then summarise. Change the question and re-run the cell to see it plan a different path.

In [ ]:
ask("What tables are in this database, and roughly what does each one hold?")

In [ ]:
ask("Pick any one table and show me the first 5 rows from it.")

## Summary

| Piece | Role |
|-------|------|
| `TOOLS` (JSON schema) | What the LLM **sees** — decides which tool to pick |
| `run_tool()` | What **you** execute when the LLM makes a call |
| The `while` loop | Keeps going until `finish_reason == "stop"` |
| `validate_sql()` | Guardrail — blocks unsafe SQL before it touches the DB |

**Key teaching moment:** the LLM doesn't know the schema until it calls `get_schema()`. You never told it "fetch the schema first" — it figured that out from the tool description.

**Try a different schema** by changing the `SCHEMA` widget — `nyctaxi`, `accuweather`, `tpch` — and re-running.

**Next:** Module 04 — Agentic Loop. Same code plus error recovery, step limits, and multi-turn memory.